#  Baseline Model: TF-IDF + Logistic Regression
This notebook handles the data preprocessing, feature combination, TF-IDF vectorization, model training, evaluation, and saving to MongoDB GridFS.

In [1]:
import pandas as pd
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

# import the GridFS handler if MongoDB is connected
from gridfs_handler import ModelGridFSHandler
has_gridfs = True

In [2]:
# 1. Load the usable dataset
dataset_path = os.path.join("..", "dataset", "usable_dataset.csv")
if not os.path.exists(dataset_path):
    dataset_path = "dataset/usable_dataset.csv"

df = pd.read_csv(dataset_path)
print(f" Dataset loaded successfully! Shape: {df.shape}")
df = df.drop_duplicates().reset_index(drop=True)   # remove duplicates

print(df.head())

 Dataset loaded successfully! Shape: (417, 9)
              Condition_name  \
0  Abdominal aortic aneurysm   
1     About aplastic anaemia   
2      Achilles tendinopathy   
3                       Acne   
4        Acute cholecystitis   

                                            Symptoms  \
0  In most cases, an AAA causes no noticeable sym...   
1  Symptoms of aplastic anaemia can vary in sever...   
2  Symptoms may vary from person to person. They ...   
3  Acne can cause different kinds of spots.\nBlac...   
4  Acute cholecystitis often causes severe pain i...   

                                              Causes  \
0  It’s not known exactly what causes the aortic ...   
1  In most cases, the cause of aplastic anaemia i...   
2  Achilles tendinopathy can occur in both active...   
3  Sebaceous glands are tiny glands found near th...   
4  The causes of acute cholecystitis can be group...   

                                            Warnings  \
0  Because AAAs usually cause n

In [3]:
# Combine all cleaned features into a single rich text column for each condition
print(" Combining all cleaned features (Symptoms, Causes, Warnings, Recommendations)...")
df['combined_features'] = (
    df['cleaned_Symptoms'] + " " + 
    df['cleaned_Causes'] + " " + 
    df['cleaned_Warnings'] + " " + 
    df['cleaned_Recommendations']
)

X = df['combined_features']
y = df['Condition_name'] # Target label (disease/condition name)

 Combining all cleaned features (Symptoms, Causes, Warnings, Recommendations)...


In [4]:
# 2. Extract features from text using TF-IDF Vectorizer
print(" Applying TF-IDF Vectorization...")
vectorizer = TfidfVectorizer(max_features=2500, ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(X)

 Applying TF-IDF Vectorization...


In [5]:
# 3. Train the Logistic Regression classifier
print(" Training the Logistic Regression model...")
clf = LogisticRegression(max_iter=1000)
clf.fit(X_tfidf,y)
print(" Training completed!")

 Training the Logistic Regression model...


m:\AMIT_AI_Diploma\final_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1469: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)


 Training completed!


In [6]:
# 4. Evaluate the model performance
preds = clf.predict(X_tfidf)
acc = accuracy_score(y, preds)
f1 = f1_score(y, preds, average='weighted', zero_division=0)

metrics = {
    "accuracy": float(acc),
    "weighted_f1": float(f1),
    "training_scope": "100%_full_dataset"
}

print(f" Baseline Results:")
print(f"   - Accuracy: {acc:.4f}")
print(f"   - Weighted F1-Score: {f1:.4f}")

 Baseline Results:
   - Accuracy: 1.0000
   - Weighted F1-Score: 1.0000


In [10]:
vectorizer = TfidfVectorizer(max_features=2500, ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(X)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_tfidf, y)

preds = clf.predict(X_tfidf)
print("Training Accuracy on 100% data:", accuracy_score(y, preds))

m:\AMIT_AI_Diploma\final_project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1469: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)


Training Accuracy on 100% data: 1.0


In [12]:
# 5. Save the model and vectorizer into MongoDB GridFS and store metadata
print(" Saving the trained model and vectorizer into MongoDB GridFS...")
handler = ModelGridFSHandler()

labels = list(clf.classes_)
handler.save_model(
    model_object=clf,
    vectorizer_object=vectorizer,
    model_name="baseline_logistic",
    model_type="Logistic Regression",
    labels=labels,
    metrics=metrics
)

print("🚀 Model successfully saved to MongoDB GridFS and ready for production!")

 Saving the trained model and vectorizer into MongoDB GridFS...
 Model 'baseline_logistic' successfully saved to GridFS with ID: 6a63fcd4f2f5468561fee9d0
🚀 Model successfully saved to MongoDB GridFS and ready for production!
